## Alt Görev 1.1: Veri Yükleme + DOI Birleştirmesi

In [1]:
# =====================================================================
# Notebook 01 — Veri Tanıma ve Keşif
# Alt Görev 1.1: Veri Yükleme + DOI Birleştirmesi
# =====================================================================
from pathlib import Path
import pandas as pd
import numpy as np

# ---------- Yollar ----------
PROJE_DIR    = Path(r"C:\Users\Kerem\Desktop\proje")
RAW_DIR      = PROJE_DIR / "data" / "raw"
SHARED_DIR   = PROJE_DIR / "data" / "shared_outputs"
SHARED_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV    = RAW_DIR / "train.csv"
TEST_CSV     = RAW_DIR / "test.csv"
ENRICHED_PQ  = RAW_DIR / "enriched.parquet"
FAILED_CSV   = RAW_DIR / "failed_dois.csv"

# ---------- DOI normalizasyonu (main.py ile bire bir) ----------
def normalize_doi(doi):
    if not isinstance(doi, str):
        return None
    doi = doi.strip().lower()
    for pfx in ("https://doi.org/", "http://doi.org/", "doi.org/", "doi:"):
        if doi.startswith(pfx):
            doi = doi[len(pfx):]
            break
    return doi if doi else None

# ---------- 1) Orijinal train/test ----------
train_raw = pd.read_csv(TRAIN_CSV)
test_raw  = pd.read_csv(TEST_CSV)

print(f"[ham] train: {len(train_raw):>6} satır | test: {len(test_raw):>6} satır")

# 'Unnamed: 0' -> 'original_idx'
for df in (train_raw, test_raw):
    if "Unnamed: 0" in df.columns:
        df.rename(columns={"Unnamed: 0": "original_idx"}, inplace=True)

# DOI normalizasyonu
train_raw["doi"] = train_raw["doi"].map(normalize_doi)
test_raw["doi"]  = test_raw["doi"].map(normalize_doi)

# ---------- 2) Tam-kopya satırları sil (train) ----------
n_before = len(train_raw)
train_dedup = train_raw.drop_duplicates(subset=["doi"], keep="first").reset_index(drop=True)
n_after = len(train_dedup)
print(f"[dedup] train: {n_before} -> {n_after} (silinen kopya: {n_before - n_after})")

# Test'te yine de kontrol edelim (rapora göre 0 bekleniyor)
test_dedup = test_raw.drop_duplicates(subset=["doi"], keep="first").reset_index(drop=True)
print(f"[dedup] test : {len(test_raw)} -> {len(test_dedup)}")

# Train↔Test arası DOI çakışması (sızıntı kontrolü, bekleme: 0)
overlap = set(train_dedup["doi"]) & set(test_dedup["doi"])
print(f"[leak-check] train ∩ test DOI çakışması: {len(overlap)} (beklenen: 0)")

# ---------- 3) Enriched.parquet ----------
enriched = pd.read_parquet(ENRICHED_PQ)
enriched["doi"] = enriched["doi"].map(normalize_doi)

# Sütun adı çakışmalarını çöz: orijinaldeki 'title'ı koru, enriched'ı '_oa' eki ile yeniden adlandır
overlap_cols = set(enriched.columns) & set(train_dedup.columns) - {"doi"}
print(f"[merge] orijinal ile enriched arası çakışan sütunlar: {sorted(overlap_cols)}")
rename_map = {c: f"{c}_oa" for c in overlap_cols}
enriched_for_merge = enriched.rename(columns=rename_map)

print(f"[enriched] {len(enriched_for_merge)} satır, {enriched_for_merge.shape[1]} sütun")

# ---------- 4) Failed DOI'ler ----------
if FAILED_CSV.exists():
    failed = pd.read_csv(FAILED_CSV)
    failed["doi"] = failed["doi"].map(normalize_doi)
    failed_set = set(failed["doi"].dropna())
    print(f"[failed] {len(failed_set)} başarısız DOI yüklendi")
    if "reason" in failed.columns:
        print("[failed] sebep dağılımı:")
        print(failed["reason"].value_counts().head(10).to_string())
else:
    failed = pd.DataFrame(columns=["doi", "reason"])
    failed_set = set()
    print("[failed] failed_dois.csv bulunamadı (atlanıyor)")

# ---------- 5) LEFT JOIN: orijinal ⋈ enriched ----------
def merge_with_enrichment(orig_df, enriched_df, label):
    merged = orig_df.merge(enriched_df, on="doi", how="left", indicator=True)
    matched = (merged["_merge"] == "both").sum()
    only_left = (merged["_merge"] == "left_only").sum()
    print(f"[merge-{label}] toplam: {len(merged)} | enrichment'lı: {matched} "
          f"({100*matched/len(merged):.1f}%) | enrichment'sız: {only_left}")
    merged.drop(columns=["_merge"], inplace=True)
    return merged

train_m = merge_with_enrichment(train_dedup, enriched_for_merge, "train")
test_m  = merge_with_enrichment(test_dedup,  enriched_for_merge, "test")

# Failed DOI'lerin train/test dağılımı
train_failed = train_dedup["doi"].isin(failed_set).sum()
test_failed  = test_dedup["doi"].isin(failed_set).sum()
print(f"[failed-dist] train tarafında başarısız DOI: {train_failed}")
print(f"[failed-dist] test  tarafında başarısız DOI: {test_failed}")
# Test'te başarısız olanlar bizim için kritik (Bahçeşehir kapsama kaybı)

# ---------- 6) split bayrağı ----------
train_m["split"] = "train"
test_m["split"]  = "test"

# ---------- 7) Kaydet ----------
out_train = SHARED_DIR / "train_step1.parquet"
out_test  = SHARED_DIR / "test_step1.parquet"
train_m.to_parquet(out_train, index=False)
test_m.to_parquet(out_test,   index=False)

print("ÖZET")
print(f"train_step1.parquet: {train_m.shape} -> {out_train}")
print(f"test_step1.parquet : {test_m.shape}  -> {out_test}")
print(f"Toplam sütun: {train_m.shape[1]} ")
print(f"\nSütunlar: {list(train_m.columns)}")

[ham] train:  22659 satır | test:    807 satır
[dedup] train: 22659 -> 21303 (silinen kopya: 1356)
[dedup] test : 807 -> 807
[leak-check] train ∩ test DOI çakışması: 0 (beklenen: 0)
[merge] orijinal ile enriched arası çakışan sütunlar: ['title']
[enriched] 21992 satır, 19 sütun
[failed] 118 başarısız DOI yüklendi
[failed] sebep dağılımı:
reason
not_found_anywhere    111
network_error           7
[merge-train] toplam: 21303 | enrichment'lı: 21190 (99.5%) | enrichment'sız: 113
[merge-test] toplam: 807 | enrichment'lı: 802 (99.4%) | enrichment'sız: 5
[failed-dist] train tarafında başarısız DOI: 113
[failed-dist] test  tarafında başarısız DOI: 5
ÖZET
train_step1.parquet: (21303, 28) -> C:\Users\Kerem\Desktop\proje\data\shared_outputs\train_step1.parquet
test_step1.parquet : (807, 28)  -> C:\Users\Kerem\Desktop\proje\data\shared_outputs\test_step1.parquet
Toplam sütun: 28 

Sütunlar: ['original_idx', 'doi', 'abstract', 'title', 'category name', 'keywords', 'university', 'modded_Q_category',

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

SHARED = Path(r"C:\Users\Kerem\Desktop\proje\data\shared_outputs")
tr = pd.read_parquet(SHARED / "train_step1.parquet")
te = pd.read_parquet(SHARED / "test_step1.parquet")

# 1) Eksik veri haritası
miss = pd.DataFrame({
    "train_missing": tr.isna().sum(),
    "train_pct": (tr.isna().mean() * 100).round(2),
    "test_missing": te.isna().sum(),
    "test_pct": (te.isna().mean() * 100).round(2),
})
miss = miss[miss["train_missing"] + miss["test_missing"] > 0].sort_values("train_pct", ascending=False)
print("EKSİK VERİ (sıfırdan büyük olanlar)")
print(miss.to_string())

# 2) Kategorik dağılımlar
cat_cols = ["modded_Q_category", "type", "language", "is_oa", "oa_status", "source", "university"]
for c in cat_cols:
    if c not in tr.columns: 
        continue
    print(f"\n--- {c} ---")
    a = tr[c].value_counts(dropna=False).head(12)
    b = te[c].value_counts(dropna=False).head(12)
    cmp = pd.DataFrame({"train": a, "test": b}).fillna(0).astype(int)
    cmp["train_%"] = (cmp["train"] / len(tr) * 100).round(1)
    cmp["test_%"] = (cmp["test"] / len(te) * 100).round(1)
    print(cmp.to_string())

# 3) Sayısal dağılımlar (medyan + kuyruklar)
num_cols = ["publication_year", "cited_by_count", "authors_count", "referenced_works_count"]
def num_summary(df, cols, label):
    rows = []
    for c in cols:
        if c not in df.columns:
            continue
        s = pd.to_numeric(df[c], errors="coerce").dropna()
        rows.append({
            "col": c, "n": len(s),
            "min": s.min(), "p25": s.quantile(.25), "median": s.median(),
            "p75": s.quantile(.75), "p95": s.quantile(.95), "p99": s.quantile(.99),
            "max": s.max(), "mean": round(s.mean(), 2),
        })
    out = pd.DataFrame(rows).set_index("col")
    print(f"\nSAYISAL — {label}")
    print(out.to_string())

num_summary(tr, num_cols, "train")
num_summary(te, num_cols, "test")

# 4) Metin uzunlukları (karakter)
print("\nMETİN UZUNLUKLARI (karakter)")
for c in ["abstract", "title", "hf_prompt"]:
    tr_len = tr[c].fillna("").str.len()
    te_len = te[c].fillna("").str.len()
    print(f"{c:10s} | train medyan={int(tr_len.median())}, p95={int(tr_len.quantile(.95))}, max={int(tr_len.max())}"
          f"  ||  test medyan={int(te_len.median())}, p95={int(te_len.quantile(.95))}, max={int(te_len.max())}")

# 5) Yıl dağılımı (cross-tab)
print("\nYAYIN YILI x SPLIT")
yr = pd.concat([
    tr["publication_year"].value_counts().rename("train"),
    te["publication_year"].value_counts().rename("test"),
], axis=1).fillna(0).astype(int).sort_index()
print(yr.to_string())

EKSİK VERİ (sıfırdan büyük olanlar)
                        train_missing  train_pct  test_missing  test_pct
keywords                         3229      15.16            62      7.68
publisher                        1858       8.72            71      8.80
journal_issn_l                    190       0.89             9      1.12
journal_name                      186       0.87             9      1.12
publication_date                  141       0.66             5      0.62
oa_status                         141       0.66             5      0.62
cited_by_count                    141       0.66             5      0.62
counts_by_year                    141       0.66             5      0.62
openalex_id                       141       0.66             5      0.62
is_oa                             141       0.66             5      0.62
language                          130       0.61             5      0.62
referenced_works_count            113       0.53             5      0.62
topics         

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

SHARED = Path(r"C:\Users\Kerem\Desktop\proje\data\shared_outputs")
tr = pd.read_parquet(SHARED / "train_step1.parquet")
te = pd.read_parquet(SHARED / "test_step1.parquet")

# 1) Mega-yazarlı makaleler (CERN tipi)
print("MEGA-YAZARLI MAKALELER")
for label, df in [("train", tr), ("test", te)]:
    n50  = (df["authors_count"] > 50).sum()
    n100 = (df["authors_count"] > 100).sum()
    n500 = (df["authors_count"] > 500).sum()
    print(f"{label}: >50 yazar = {n50}, >100 = {n100}, >500 = {n500}")

mega = tr[tr["authors_count"] > 100][["doi", "authors_count", "journal_name", 
                                       "category name", "modded_Q_category", 
                                       "cited_by_count"]]
print(f"\nTrain'de 100+ yazarlı {len(mega)} makale — dergi dağılımı:")
print(mega["journal_name"].value_counts().head(8).to_string())
print("\nQ dağılımı:")
print(mega["modded_Q_category"].value_counts().to_string())

# 2) Title'larda HTML etiketleri
html_pat = re.compile(r"<[^>]+>")
tr_html = tr["title"].fillna("").str.contains(html_pat, regex=True)
te_html = te["title"].fillna("").str.contains(html_pat, regex=True)
print(f"\nHTML ETİKETLİ BAŞLIK: train {tr_html.sum()} ({tr_html.mean()*100:.1f}%), "
      f"test {te_html.sum()} ({te_html.mean()*100:.1f}%)")

# Hangi etiketler geçiyor?
sample_tags = tr.loc[tr_html, "title"].head(8).tolist()
all_tags = set()
for t in tr.loc[tr_html, "title"].fillna(""):
    all_tags.update(html_pat.findall(t))
print(f"Görülen tag sayısı: {len(all_tags)}, en sık olanlardan: ")
print(pd.Series([m for t in tr.loc[tr_html, "title"].fillna("") for m in html_pat.findall(t)]).value_counts().head(10).to_string())

# 3) Aşırı uzun abstract'lar
long_abs = tr["abstract"].fillna("").str.len() > 5000
print(f"\nUZUN ABSTRACT (>5000 karakter): train {long_abs.sum()}, test {(te['abstract'].fillna('').str.len()>5000).sum()}")
# Bu makalelerin türü ne?
if long_abs.sum() > 0:
    print(tr.loc[long_abs, "type"].value_counts().to_string())

# 4) Çok kısa abstract'lar (potansiyel kalitesiz veri)
short_abs = tr["abstract"].fillna("").str.len() < 100
print(f"\nKISA ABSTRACT (<100 karakter): train {short_abs.sum()}, test {(te['abstract'].fillna('').str.len()<100).sum()}")

# 5) Letonca makaleler — bunlar gerçekten Letonca mı?
lv = tr[tr["language"] == "lv"][["title", "journal_name", "category name", "university"]].head(5)
print(f"\nLetonca etiketli 5 örnek (gerçekten Letonca mı?):")
for i, r in lv.iterrows():
    print(f"  - title: {r['title'][:80]}")
    print(f"    journal: {r['journal_name']}")
    print()

# 6) Sıfır atıflı makalelerin profili
zero_cite = tr[tr["cited_by_count"] == 0]
print(f"SIFIR ATIFLI: train {len(zero_cite)} ({len(zero_cite)/len(tr)*100:.1f}%), "
      f"test {(te['cited_by_count']==0).sum()}")
print(f"  yıl dağılımı: {zero_cite['publication_year'].value_counts().sort_index().to_dict()}")
print(f"  Q dağılımı: {zero_cite['modded_Q_category'].value_counts().to_dict()}")

# 7) Yayın tarihi vs counts_by_year tutarlılığı (atıf zaman serisi sağlık kontrolü)
import json
def first_cite_year(cby_str):
    if not isinstance(cby_str, str): return None
    try:
        arr = json.loads(cby_str)
        if not arr: return None
        return min(c["year"] for c in arr if "year" in c)
    except: return None

sample = tr.dropna(subset=["counts_by_year", "publication_year"]).sample(min(2000, len(tr)), random_state=0)
sample["first_cite_year"] = sample["counts_by_year"].map(first_cite_year)
sample["pub_year"] = sample["publication_year"].astype("Int64")
mask = sample["first_cite_year"].notna()
weird = sample[mask & (sample["first_cite_year"] < sample["pub_year"])]
print(f"\nilk atıf yılı < yayın yılı olan makale (örneklemde): {len(weird)}")
# Bu olabiliyor (preprint vs final yayın tarihi farkı), ama yaygın olmamalı

# 8) authors_count = 0 olan makaleler (anomali)
zero_auth = tr[tr["authors_count"] == 0]
print(f"\nSIFIR YAZARLI: train {len(zero_auth)}, test {(te['authors_count']==0).sum()}")
if len(zero_auth) > 0:
    print(zero_auth[["doi", "title", "type"]].head(5).to_string(index=False))

MEGA-YAZARLI MAKALELER
train: >50 yazar = 659, >100 = 503, >500 = 391
test: >50 yazar = 8, >100 = 0, >500 = 0

Train'de 100+ yazarlı 503 makale — dergi dağılımı:
journal_name
Journal of High Energy Physics                181
The European Physical Journal C                77
Physics Letters B                              42
Physical review. D/Physical review. D.         32
Physical Review Letters                        32
Journal of Instrumentation                     18
Biointerface Research in Applied Chemistry     10
Nature Physics                                  5

Q dağılımı:
modded_Q_category
Q1    338
Q2    137
Q4     27
Q3      1

HTML ETİKETLİ BAŞLIK: train 2125 (10.0%), test 48 (5.9%)
Görülen tag sayısı: 21, en sık olanlardan: 
<i>              2434
</i>             2434
<sub>             710
</sub>            710
<SUP>             310
</SUP>            310
</mml:msqrt>        2
<mml:msqrt>         2
</mml:mspace>       2
<overbar>           1

UZUN ABSTRACT (>5000 karakter):

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

SHARED = Path(r"C:\Users\Kerem\Desktop\proje\data\shared_outputs")
tr = pd.read_parquet(SHARED / "train_step1.parquet")
te = pd.read_parquet(SHARED / "test_step1.parquet")

# Q için bir tek "concentration" metriği yazalım
def q_concentration(df, col, min_count=10):
    """Bir kategorik sütun için, her unique değerin Q dağılımındaki tepe oranını ölç."""
    g = df.dropna(subset=[col]).groupby(col)["modded_Q_category"]
    rows = []
    for val, sub in g:
        n = len(sub)
        if n < min_count: 
            continue
        top_q_share = sub.value_counts(normalize=True).max()
        rows.append({"value": val, "n": n, "top_q_share": top_q_share, 
                     "top_q": sub.value_counts().idxmax()})
    out = pd.DataFrame(rows)
    if len(out) == 0:
        return out, None
    summary = {
        "n_unique_with_data": len(out),
        "median_top_q_share": out["top_q_share"].median(),
        "pct_above_90": (out["top_q_share"] >= 0.90).mean() * 100,
        "pct_above_75": (out["top_q_share"] >= 0.75).mean() * 100,
    }
    return out, summary

# 1) journal_name (beklenen: aşırı sızıntı)
_, s = q_concentration(tr, "journal_name", min_count=10)
print("journal_name (n>=10):", s)

# 2) journal_issn_l
_, s = q_concentration(tr, "journal_issn_l", min_count=10)
print("journal_issn_l (n>=10):", s)

# 3) publisher
_, s = q_concentration(tr, "publisher", min_count=20)
print("publisher (n>=20):", s)

# 4) university
_, s = q_concentration(tr, "university", min_count=50)
print("university (n>=50):", s)

# 5) category name (ham haliyle, pipe'lı kombinasyonlar dahil)
_, s = q_concentration(tr, "category name", min_count=20)
print("category name (n>=20):", s)

# 6) category name (pipe'la bölünüp ana kategoriye düşürüldükten sonra)
tr_main_cat = tr.assign(main_cat=tr["category name"].fillna("").str.split("|").str[0].str.strip())
_, s = q_concentration(tr_main_cat, "main_cat", min_count=50)
print("main_cat (pipe sonrası ilk kategori, n>=50):", s)

# 7) oa_status
_, s = q_concentration(tr, "oa_status", min_count=50)
print("oa_status:", s)

# 8) type
_, s = q_concentration(tr, "type", min_count=20)
print("type:", s)

# Test setinde "görünmeyen" değer kontrolü
print("\nTEST'TE GÖRÜNMEYEN DEĞER ANALİZİ")
for col in ["category name", "journal_name", "publisher", "main_cat" if False else "category name"]:
    pass

# Test'teki kategoriler train'de var mı?
te_main_cat = te.assign(main_cat=te["category name"].fillna("").str.split("|").str[0].str.strip())
tr_set_full = set(tr["category name"].dropna())
te_set_full = set(te["category name"].dropna())
tr_set_main = set(tr_main_cat["main_cat"])
te_set_main = set(te_main_cat["main_cat"])
print(f"category name (ham): test'te {len(te_set_full)} unique, train'de yok = {len(te_set_full - tr_set_full)}")
print(f"main_cat (pipe sonrası): test'te {len(te_set_main)} unique, train'de yok = {len(te_set_main - tr_set_main)}")

# Test'teki dergiler train'de var mı? (Bu kritik — Bahçeşehir farklı dergilere mi yayın yapıyor?)
tr_journals = set(tr["journal_name"].dropna())
te_journals = set(te["journal_name"].dropna())
overlap_j = te_journals & tr_journals
print(f"journal: test'te {len(te_journals)} unique, train'de görünen = {len(overlap_j)} ({len(overlap_j)/len(te_journals)*100:.1f}%)")
unseen_test_papers = te[~te["journal_name"].isin(tr_journals) & te["journal_name"].notna()]
print(f"  test'te train'de hiç görünmemiş bir dergide yayınlanan makale sayısı: {len(unseen_test_papers)}")

# Q1/Q4 örnek kategoriler raporu (önceki keşifte gördüğümüzü tekrar doğrula)
print("\nKATEGORİ-Q İLİŞKİSİ (n>=100, en uçtaki kategoriler):")
df_cat = tr.groupby("category name").agg(
    n=("doi","count"),
    q1=("modded_Q_category", lambda x: (x=="Q1").mean()),
    q4=("modded_Q_category", lambda x: (x=="Q4").mean()),
).reset_index()
df_cat = df_cat[df_cat["n"]>=100].sort_values("q1", ascending=False)
print("En Q1-yoğun 5:")
print(df_cat.head(5).to_string(index=False))
print("En Q4-yoğun 5:")
print(df_cat.sort_values("q4", ascending=False).head(5).to_string(index=False))

# Final sızıntı kararı tablosu
print("\nSIZINTI KARARLARI (tasarım için)")
print("-" * 60)
decisions = pd.DataFrame([
    {"feature": "journal_name", "verdict": "BLOCK", "neden": "Q dergiden türetildi, tautoloji"},
    {"feature": "journal_issn_l", "verdict": "BLOCK", "neden": "journal_name ile bire bir"},
    {"feature": "publisher", "verdict": "BLOCK", "neden": "yayıncı seviyesinde de güçlü sinyal, aynı tür sızıntı"},
    {"feature": "cited_by_count", "verdict": "BLOCK (Modül A)", "neden": "yayın anında bilinmeyen, gelecek bilgi"},
    {"feature": "university (raw)", "verdict": "BLOCK (base)", "neden": "OOD'da anlamsız (Bahçeşehir train'de yok)"},
    {"feature": "univ_freq encoding", "verdict": "AYRI SENARYO", "neden": "B senaryosunda kıyas için"},
    {"feature": "category name", "verdict": "ALLOW", "neden": "içerik sinyali, OOD'da çoğu görünüyor"},
    {"feature": "main_cat", "verdict": "ALLOW", "neden": "daha temiz, %96 OOD kapsama"},
    {"feature": "oa_status / is_oa", "verdict": "ALLOW", "neden": "zayıf sinyal, OOD güvenli"},
    {"feature": "type", "verdict": "ALLOW", "neden": "zayıf sinyal, OOD güvenli"},
    {"feature": "language (gruplu)", "verdict": "ALLOW", "neden": "en/tr/other şeklinde, OOD güvenli"},
    {"feature": "abstract / title (içerik)", "verdict": "ALLOW", "neden": "asıl modelimizin omurgası"},
])
print(decisions.to_string(index=False))

journal_name (n>=10): {'n_unique_with_data': 455, 'median_top_q_share': 1.0, 'pct_above_90': 100.0, 'pct_above_75': 100.0}
journal_issn_l (n>=10): {'n_unique_with_data': 455, 'median_top_q_share': 1.0, 'pct_above_90': 100.0, 'pct_above_75': 100.0}
publisher (n>=20): {'n_unique_with_data': 85, 'median_top_q_share': 0.7123745819397993, 'pct_above_90': 36.470588235294116, 'pct_above_75': 44.70588235294118}
university (n>=50): {'n_unique_with_data': 15, 'median_top_q_share': 0.3333333333333333, 'pct_above_90': 0.0, 'pct_above_75': 0.0}
category name (n>=20): {'n_unique_with_data': 225, 'median_top_q_share': 0.6046511627906976, 'pct_above_90': 16.0, 'pct_above_75': 29.777777777777775}
main_cat (pipe sonrası ilk kategori, n>=50): {'n_unique_with_data': 120, 'median_top_q_share': 0.45483193277310924, 'pct_above_90': 1.6666666666666667, 'pct_above_75': 3.3333333333333335}
oa_status: {'n_unique_with_data': 6, 'median_top_q_share': 0.36158779576587796, 'pct_above_90': 0.0, 'pct_above_75': 0.0}
t